# Lab 11: Check, release, and roll back an ACT checkpoint

**Goal:** Check where a checkpoint came from. Check its hashes. Write a release record. Decide whether to keep or roll back a candidate. Prove that the rollback can be undone.

**Inputs:** Tracked compact records under `artifacts/reference/`, tracked manifests, and optional local checkpoint files. **Compute:** CPU. **Cloud cost:** US$0. No paid job ran for this lab. Start Jupyter from the repository root. Read the [lab guide](README.md) and the [operations guide](../../docs/operations.md).

**Limit:** This lab reads saved results. It does not train, evaluate, or start a simulator. It writes only to a temporary directory. A rollback here is a pointer change in that directory. It does not change a real checkpoint.

## 1. Find the repository root and load tracked records

The notebook looks for `manifests/act_references.yaml` from the current directory upward. Then it loads each tracked record once. Every later cell uses these names.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import tempfile
from pathlib import Path

import yaml

from vla_lab.evaluation.paired import exact_mcnemar, wilson_interval


def find_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the repository.")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_json(relative):
    return json.loads((WORKSPACE / relative).read_text())


def read_rows(relative):
    return [json.loads(line) for line in (WORKSPACE / relative).read_text().splitlines()]


WORKSPACE = find_root()
act_manifest = yaml.safe_load((WORKSPACE / "manifests/act_references.yaml").read_text())
transport = read_json("artifacts/reference/act/transport.json")
final_report = read_json("artifacts/reference/comparison/final/three_method_report.json")
recovery = read_json("artifacts/reference/data_quality/recovery_reference.json")
splits = yaml.safe_load((WORKSPACE / "configs/eval/splits.yaml").read_text())
print("Workspace:", WORKSPACE)
print("Tracked records loaded: 5")

## 2. Check the hash chain in tracked records

One model file appears in three tracked records. Four values must be the same SHA-256 hash:

1. The reference manifest for the `bowl_plate` ACT checkpoint.
2. The frozen three-method report (the ACT model used in the final comparison).
3. The Lab 8 recovery record, `base_model_sha256`.
4. The Lab 8 recovery record, `teacher_model_sha256`.

The cell then checks the comparison config hash in the report. A match shows that the records name the same file. It does not show that the file is a good policy.

In [ ]:
bowl_plate = act_manifest["tasks"]["bowl_plate"]
current_model_sha = bowl_plate["checkpoint_files_sha256"]["model.safetensors"]
chain = {
    "act_references.yaml": current_model_sha,
    "three_method_report.json (ACT)": final_report["methods"]["act"]["model_sha256"],
    "recovery_reference.json base": recovery["base_model_sha256"],
    "recovery_reference.json teacher": recovery["teacher_model_sha256"],
}
for name, value in chain.items():
    print(f"{value[:16]}  {name}")
assert len(set(chain.values())) == 1

comparison_config = WORKSPACE / "configs/experiments/comparison.yaml"
assert sha256(comparison_config) == final_report["comparison_config_sha256"]
print("Comparison config hash matches the report:", final_report["comparison_config_sha256"][:16])

## 3. Check local files when they exist

Checkpoint files, the transfer archive, and the recovery checkpoint are large. Git ignores them. They live under `.local-data/`. On a fresh clone they can be absent. This cell hashes each file that exists. It reports each missing file as `absent`. It fails only on a hash mismatch.

A `match` result proves that the local file equals the tracked hash. An `absent` result proves nothing. Restore the files with the commands in the [operations guide](../../docs/operations.md) when you need them.

In [ ]:
def check_local(label, path, expected):
    path = WORKSPACE / path
    if not path.is_file():
        return label, "absent"
    actual = sha256(path)
    assert actual == expected, f"Hash mismatch for {label}: {actual}"
    return label, "match"


results = []
for filename, expected in bowl_plate["checkpoint_files_sha256"].items():
    results.append(check_local(f"bowl_plate/{filename}", f"{bowl_plate['checkpoint_path']}/{filename}", expected))
results.append(check_local("transfer archive", transport["archive_path"], transport["archive_sha256"]))
candidate_checkpoint = ".local-data/runs/act-bowl-plate-recovery-200-tick-aligned/checkpoint-000200/model.safetensors"
results.append(check_local("tick-aligned candidate model", candidate_checkpoint, recovery["tick_aligned_model_sha256"]))
for label, status in results:
    print(f"{status:>6}  {label}")
local_matches = sum(status == "match" for _, status in results)
print(f"{local_matches} of {len(results)} local files checked; the rest are absent.")

## 4. Check the provenance of the candidate

The current release is the 200-step ACT checkpoint for `bowl_plate`. The candidate is the Lab 8 tick-aligned recovery continuation. It starts from the current weights. It trains 200 more steps on 5 policy-generated recordings. It has no human demonstrations.

Read the tracked record and state three facts: the base hash, the recording source type, and the states used to collect data. Then check that collection states and assessment states do not overlap.

In [ ]:
assert recovery["base_model_sha256"] == current_model_sha
assert recovery["source_type"] == "policy_generated" and recovery["human_demonstrations"] == 0
collection, assessment = set(recovery["collection_state_ids"]), set(recovery["assessment_state_ids"])
assert not collection & assessment
development = set(splits["tasks"]["bowl_plate"]["development_state_ids"])
assert collection | assessment <= development  # no final state was used
print("Base hash:", recovery["base_model_sha256"][:16])
print("Candidate hash:", recovery["tick_aligned_model_sha256"][:16])
print("Source:", recovery["source_type"], "| recordings:", recovery["accepted_recordings"])
print("Collection states:", sorted(collection), "| assessment states:", sorted(assessment))
print("Recovery scope:", recovery["scope"])

## 5. Write release records

A release record names one checkpoint and the evidence for it. This lab writes two records into a temporary directory. The records use only values that tracked files prove. A missing value is `null`. It is never guessed.

The repository state can be dirty. The record stores the commit and a dirty flag. The cell does not require a clean tree.

In [ ]:
def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=WORKSPACE, capture_output=True, text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return None


lock_sha = sha256(WORKSPACE / "uv.lock")
status = git("status", "--porcelain")
git_state = {"commit": git("rev-parse", "HEAD"), "dirty": None if status is None else bool(status)}

base_rows = read_rows(recovery["results"]["base"]["episodes_file"])
cand_rows = read_rows(recovery["results"]["tick_aligned"]["episodes_file"])
evidence = {
    "base": recovery["results"]["base"]["episodes_file"],
    "candidate": recovery["results"]["tick_aligned"]["episodes_file"],
}

def release_record(release_id, model_sha, files, rows, episodes_file, parent):
    return {
        "schema_version": 1,
        "release_id": release_id,
        "task_alias": "bowl_plate",
        "model_sha256": model_sha,
        "checkpoint_files_sha256": files,
        "parent_release": parent,
        "evaluation": {
            "partition": "development",
            "state_ids": [row["state_id"] for row in rows],
            "successes": sum(row["success"] for row in rows),
            "attempts": len(rows),
            "episodes_file": episodes_file,
            "episodes_sha256": sha256(WORKSPACE / episodes_file),
        },
        "repository": git_state,
        "uv_lock_sha256": lock_sha,
        "paid_cloud_cost_usd": 0,
        "limits": [
            "One training seed.",
            "15 development states; no final state opened.",
            "Source demonstration overlap with simulator states is unresolved.",
        ],
    }


release_a = release_record("bowl_plate-act-200", current_model_sha, bowl_plate["checkpoint_files_sha256"],
                           base_rows, evidence["base"], None)
release_b = release_record("bowl_plate-act-200-tick-aligned", recovery["tick_aligned_model_sha256"],
                           {"model.safetensors": recovery["tick_aligned_model_sha256"]},
                           cand_rows, evidence["candidate"], "bowl_plate-act-200")

SCRATCH = Path(tempfile.mkdtemp(prefix="lab11-release-"))
(SCRATCH / "records").mkdir()
for record in (release_a, release_b):
    (SCRATCH / "records" / f"{record['release_id']}.json").write_text(json.dumps(record, indent=2, sort_keys=True) + "\n")
print("Scratch directory:", SCRATCH)
for record in (release_a, release_b):
    ev = record["evaluation"]
    print(record["release_id"], f"{ev['successes']}/{ev['attempts']}", "model", record["model_sha256"][:12])
print("Git commit:", git_state["commit"][:12] if git_state["commit"] else None, "| dirty:", git_state["dirty"])

## 6. Decide with a rule written before the result

The rule is fixed here, before the paired counts are printed:

1. Reject the candidate if any hash or state-ID check fails.
2. Reject the candidate if any attempt has an error.
3. Promote the candidate only if it has **more** successes than the current release on the same ordered states.
4. Otherwise keep the current release.

Equal success counts mean no promotion. One state can change each way. That does not show an improvement.

In [ ]:
def decide(current_rows, candidate_rows):
    if [r["state_id"] for r in current_rows] != [r["state_id"] for r in candidate_rows]:
        return "reject", "state IDs differ"
    if [r["reset_seed"] for r in current_rows] != [r["reset_seed"] for r in candidate_rows]:
        return "reject", "reset seeds differ"
    if any(r.get("error") for r in (*current_rows, *candidate_rows)):
        return "reject", "an attempt has an error"
    cur = sum(r["success"] for r in current_rows)
    cand = sum(r["success"] for r in candidate_rows)
    return ("promote" if cand > cur else "keep"), f"candidate {cand} vs current {cur}"


decision, reason = decide(base_rows, cand_rows)
current_only = [a["state_id"] for a, b in zip(base_rows, cand_rows, strict=True) if a["success"] and not b["success"]]
candidate_only = [a["state_id"] for a, b in zip(base_rows, cand_rows, strict=True) if b["success"] and not a["success"]]
n = len(base_rows)
low, high = wilson_interval(sum(r["success"] for r in cand_rows), n)
print("Decision:", decision, "-", reason)
print("Current-only success states:", current_only, "| candidate-only success states:", candidate_only)
print("Exact McNemar p:", round(exact_mcnemar(len(current_only), len(candidate_only)), 4))
print(f"Candidate Wilson 95% interval: [{low:.3f}, {high:.3f}]")
assert current_only == recovery["paired_base_only_state_ids"]
assert candidate_only == recovery["paired_continuation_only_state_ids"]
assert decision == "keep"

## 7. Show that a rollback is reversible

A release pointer is a small file named `current.json`. An append-only `history.jsonl` records each change. Records are never deleted or edited.

The cell does three things in the scratch directory:

1. It promotes the candidate. This is a what-if step. The rule above did not choose it.
2. It rolls back to the parent release.
3. It rolls forward again.

After each step, it checks that both release records keep their original hashes.

In [ ]:
def current_release():
    return json.loads((SCRATCH / "current.json").read_text())["release_id"]


def switch(release_id, action):
    record_path = SCRATCH / "records" / f"{release_id}.json"
    record = json.loads(record_path.read_text())
    previous = current_release() if (SCRATCH / "current.json").exists() else None
    (SCRATCH / "current.json").write_text(json.dumps({"release_id": release_id, "model_sha256": record["model_sha256"]}) + "\n")
    with (SCRATCH / "history.jsonl").open("a") as stream:
        stream.write(json.dumps({"action": action, "from": previous, "to": release_id}) + "\n")


record_hashes = {p.name: sha256(p) for p in (SCRATCH / "records").iterdir()}
switch("bowl_plate-act-200", "initial")
assert current_release() == "bowl_plate-act-200"
switch("bowl_plate-act-200-tick-aligned", "promote (what-if)")
assert current_release() == "bowl_plate-act-200-tick-aligned"
parent = json.loads((SCRATCH / "records" / "bowl_plate-act-200-tick-aligned.json").read_text())["parent_release"]
switch(parent, "rollback")
assert current_release() == "bowl_plate-act-200"
pointer = json.loads((SCRATCH / "current.json").read_text())
assert pointer["model_sha256"] == current_model_sha  # same file hash as before
switch("bowl_plate-act-200-tick-aligned", "roll forward")
switch(parent, "rollback")
assert {p.name: sha256(p) for p in (SCRATCH / "records").iterdir()} == record_hashes  # no record changed
for line in (SCRATCH / "history.jsonl").read_text().splitlines():
    print(line)
print("Final pointer:", current_release())

## 8. Optional steps: GPU and cloud

**These steps are optional. The notebook does not run them by default.** The flag below is `False`.

- **Optional GPU check.** Set `RUN_GPU_CHECK = True` only on the prepared local host with a GPU and the LIBERO assets. It runs `robot-lab doctor --live`. This check needs a terminal that can see the GPU. A restricted shell cannot see it.
- **Optional cloud step.** No cell runs a cloud command. The repository has no real Brev allocation, no remote copy, and no independent provider stop test. A fake-provider test passes. No paid work ran for this lab or for this project. Read [the cloud record](../../docs/cloud.md) before any paid step. Ask the user before you start one.

In [ ]:
RUN_GPU_CHECK = False  # optional; needs the local GPU host

if RUN_GPU_CHECK:
    done = subprocess.run([str(WORKSPACE / ".venv/bin/robot-lab"), "doctor", "--live"],
                          cwd=WORKSPACE, capture_output=True, text=True, timeout=600)
    print("exit code:", done.returncode)
    print(done.stdout[-1500:])
else:
    print("Optional GPU check skipped. Cloud steps: none run. Cloud cost: US$0.")

## 9. Clean up and state the limits

Remove the scratch directory. Then confirm that no tracked file changed because of this notebook.

You proved these facts:

- Three tracked records (four values) name the same current model hash.
- Local files that exist match their tracked hashes. Missing files stay `absent`.
- The candidate came from the current weights and 5 policy-generated recordings on development states.
- The rule gave `keep` because the candidate had the same success count as the current release.
- A rollback and a roll forward changed only a pointer. No record changed.

You did not prove these facts:

- You did not run a policy or a simulator.
- You did not test the candidate on final states.
- You did not run a cloud job. You did not measure a cloud cost.
- You did not verify unattended cloud stop controls. You did not remove the source-data overlap limit.

Next, read the [exercise](exercise.md). Read the [solution guide](../../solutions/11-deployment.md) after you save your answer.

In [ ]:
shutil.rmtree(SCRATCH)
assert not SCRATCH.exists()
print("Scratch directory removed.")

## 10. Write a career assessment memo

Finish the course with a short personal memo. Answer six questions: Which work was interesting? Which repeated tasks were acceptable? Was a VLA better than a narrow imitation policy for this fixed task? What data would a real deployment need? Which integration tasks used skills that you already have? What result would change your decision?

Use the [exercise](exercise.md) for the questions and the [example memo](../../solutions/11-deployment.md) only after you write your own answer. State a measured course result and a limit. Personal preferences have no model answer.
